# Genesis — fine-tuning de GLiNER pour les CV (FR/EN)

**Exécution :** Colab → *Exécution → Modifier le type d'exécution* → **GPU T4**. Durée : ~45 min.

1. Icône 🔑 *Secrets* → `HF_TOKEN` (jeton *Write*) **et activez « Accès au notebook »**.
2. Exécutez la **cellule 1 seule** : elle installe les bonnes versions puis **redémarre la session** automatiquement (message « session plantée » normal).
3. Puis *Exécution → Exécuter la cellule sélectionnée et les suivantes* à partir de la cellule 2.
4. Le modèle n'est publié (en **privé**) que s'il **progresse sur 12 CV réalistes jamais vus**. Il est alors exporté en ONNX fp16 : sur le PC,
   `py -3 -I scripts/download_models.py --gliner <votre-pseudo>/genesis-gliner-cv-onnx` puis redémarrez Genesis.

**Données** (choisies après vérification) :
- CV synthétiques FR/EN générés avec annotations exactes (mises en page variées, majuscules, puces, tableaux) ;
- textes réels de CV `Mehyaar/Annotated_NER_PDF_Resumes` (MIT) **ré-annotés par le modèle de base** (prédictions très sûres uniquement) pour qu'il n'oublie pas ce qu'il sait déjà.

Écartés : `yashpwr/resume-ner-training-data` (conversations, aucune annotation d'entités), annotations d'origine de Mehyaar (bruitées : « Education », « Work » étiquetés compétences), WikiNEuRal (licence non commerciale).

In [ ]:
# 1) Versions compatibles entre elles (Colab récent : huggingface_hub 1.x casse gliner 0.2.16).
import importlib.metadata as md, os
WANT = {"gliner": "0.2.16", "transformers": "4.51.3", "huggingface_hub": "0.30.2"}
def ver(p):
    try: return md.version(p)
    except md.PackageNotFoundError: return None
if any(ver(p) != v for p, v in WANT.items()):
    !pip -q install gliner==0.2.16 transformers==4.51.3 huggingface_hub==0.30.2 faker==33.1.0 onnx onnxconverter-common
    print("Installé. Redémarrage de la session… puis exécutez à partir de la cellule 2.")
    os.kill(os.getpid(), 9)
else:
    print("Versions OK :", {p: ver(p) for p in WANT})

In [ ]:
# 2) Connexion Hugging Face
import os, random, re, json, itertools, collections, zipfile
random.seed(42)
HF_TOKEN = None
try:
    from google.colab import userdata; HF_TOKEN = userdata.get("HF_TOKEN")  # nécessite « Accès au notebook »
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient; HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("Jeton Hugging Face (hf_..., Write) : ").strip()
assert HF_TOKEN.startswith("hf_"), "Jeton Hugging Face invalide"
from huggingface_hub import login, whoami, hf_hub_download
login(HF_TOKEN); USER = whoami()["name"]; print("Connecté :", USER)
LABELS = ["person", "organization", "job title", "city"]   # exactement les étiquettes utilisées par Genesis
BASE_MODEL = "urchade/gliner_multi-v2.1"
TARGET_REPO = f"{USER}/genesis-gliner-cv"
WORD_RE = re.compile(r"\w+(?:[-_]\w+)*|\S")              # découpage identique à GLiNER
def words_of(text): return [(m.group(), m.start(), m.end()) for m in WORD_RE.finditer(text)]

## 3. Jeu de référence : 12 CV réalistes annotés à la main (jamais vus à l'entraînement)

In [ ]:
GOLD_RAW = [
("""Aminata DIALLO
Développeuse Full-Stack
Brazzaville, Congo
EXPÉRIENCE PROFESSIONNELLE
Développeuse Full-Stack — Société Congolaise de Logiciels (janv. 2021 - aujourd'hui)
Développeuse Web — Agence Digitale Pointe-Noire (2019 - 2020)""",
 [("person","Aminata DIALLO"),("job title","Développeuse Full-Stack"),("city","Brazzaville"),("organization","Société Congolaise de Logiciels"),("job title","Développeuse Web"),("organization","Agence Digitale Pointe-Noire")]),
("""John Mbemba
Senior Accountant – Kinshasa
WORK EXPERIENCE
Senior Accountant, Ecobank, Kinshasa (2018 – present)
Junior Auditor, Deloitte, Lubumbashi (2015 – 2018)""",
 [("person","John Mbemba"),("job title","Senior Accountant"),("city","Kinshasa"),("organization","Ecobank"),("job title","Junior Auditor"),("organization","Deloitte"),("city","Lubumbashi")]),
("""CURRICULUM VITAE
NOM : NGOMA PRENOM : Sandrine
Adresse : Pointe-Noire
2016-2023 : Secrétaire de direction chez TotalEnergies EP Congo
2014-2016 : Réceptionniste à l'Hôtel Atlantic Palace""",
 [("person","NGOMA"),("person","Sandrine"),("city","Pointe-Noire"),("job title","Secrétaire de direction"),("organization","TotalEnergies EP Congo"),("job title","Réceptionniste"),("organization","Hôtel Atlantic Palace")]),
("""Moussa Traoré | Chef de projet IT | Dakar
• Chef de projet IT, Orange Sénégal, depuis 2020
• Analyste fonctionnel, Wave Mobile Money, 2017-2020""",
 [("person","Moussa Traoré"),("job title","Chef de projet IT"),("city","Dakar"),("organization","Orange Sénégal"),("job title","Analyste fonctionnel"),("organization","Wave Mobile Money")]),
("""Grâce Okemba
Infirmière diplômée d'État
Centre Hospitalier Universitaire de Brazzaville — Infirmière en pédiatrie (2019-2024)
Clinique Netcare, Pointe-Noire — Aide-soignante (2016-2019)""",
 [("person","Grâce Okemba"),("job title","Infirmière diplômée d'État"),("organization","Centre Hospitalier Universitaire de Brazzaville"),("job title","Infirmière en pédiatrie"),("organization","Clinique Netcare"),("city","Pointe-Noire"),("job title","Aide-soignante")]),
("""PIERRE-ALAIN MOUKOKO
Responsable logistique
Douala - Cameroun
Bolloré Transport & Logistics : Responsable logistique (2018 à ce jour)
CFAO Motors : Gestionnaire de stock (2014 à 2018)""",
 [("person","PIERRE-ALAIN MOUKOKO"),("job title","Responsable logistique"),("city","Douala"),("organization","Bolloré Transport & Logistics"),("organization","CFAO Motors"),("job title","Gestionnaire de stock")]),
("""Fatou Ndiaye
Data Analyst
Abidjan
Expériences
2022 – Data Analyst – Société Générale Côte d'Ivoire
2020 – Stagiaire BI – MTN Côte d'Ivoire""",
 [("person","Fatou Ndiaye"),("job title","Data Analyst"),("city","Abidjan"),("organization","Société Générale Côte d'Ivoire"),("job title","Stagiaire BI"),("organization","MTN Côte d'Ivoire")]),
("""Jean-Baptiste Kouka
Ingénieur génie civil
Libreville, Gabon
Ingénieur travaux chez Colas Gabon (2017-2023)
Conducteur de travaux chez Sogea Satom (2013-2017)""",
 [("person","Jean-Baptiste Kouka"),("job title","Ingénieur génie civil"),("city","Libreville"),("job title","Ingénieur travaux"),("organization","Colas Gabon"),("job title","Conducteur de travaux"),("organization","Sogea Satom")]),
("""Sarah Martin
Chargée de recrutement — Lyon
Chargée de recrutement, Adecco, Lyon (2021-2024)
Assistante RH, Groupe SEB, Écully (2018-2021)""",
 [("person","Sarah Martin"),("job title","Chargée de recrutement"),("city","Lyon"),("organization","Adecco"),("job title","Assistante RH"),("organization","Groupe SEB"),("city","Écully")]),
("""Kevin Loubaki
Commercial terrain
Brazzaville
Commercial terrain — Brasseries du Congo (2019-2024)
Vendeur — Supermarché Casino Mbota (2016-2019)""",
 [("person","Kevin Loubaki"),("job title","Commercial terrain"),("city","Brazzaville"),("organization","Brasseries du Congo"),("job title","Vendeur"),("organization","Supermarché Casino Mbota")]),
("""Emily Johnson
Marketing Manager
London, United Kingdom
Marketing Manager at Unilever, London (2019 – Present)
Brand Executive at Diageo, Edinburgh (2015 – 2019)""",
 [("person","Emily Johnson"),("job title","Marketing Manager"),("city","London"),("organization","Unilever"),("job title","Brand Executive"),("organization","Diageo"),("city","Edinburgh")]),
("""Mme Clarisse Mabiala
Juriste d'affaires
Kinshasa
Juriste d'affaires — Cabinet Emery Mukendi Wafwana & Associates (depuis 2021)
Assistante juridique — Rawbank (2018-2021)""",
 [("person","Clarisse Mabiala"),("job title","Juriste d'affaires"),("city","Kinshasa"),("organization","Cabinet Emery Mukendi Wafwana & Associates"),("job title","Assistante juridique"),("organization","Rawbank")]),
]

def to_example(text, ents):
    w = words_of(text)
    spans = []
    for lab, s in ents:
        for m in re.finditer(re.escape(s), text):
            idx = [i for i, (_, a, b) in enumerate(w) if a >= m.start() and b <= m.end()]
            if idx: spans.append([idx[0], idx[-1], lab])
    spans = [list(x) for x in {tuple(s) for s in spans}]
    return {"tokenized_text": [t for t, _, _ in w], "ner": spans}

gold = [to_example(t, e) for t, e in GOLD_RAW]
print(len(gold), "CV de référence,", sum(len(g["ner"]) for g in gold), "entités")

## 4. CV synthétiques FR/EN (annotations exactes, mises en page variées)

In [ ]:
from faker import Faker
fk = {"fr": Faker(["fr_FR", "fr_FR", "fr_CA", "fr_CH"]), "en": Faker(["en_US", "en_GB"])}
ORGS = {"fr": ["Société Congolaise de Logiciels", "Orange Côte d'Ivoire", "Banque Atlantique", "TotalEnergies", "Airtel Congo", "MTN Cameroun",
               "Société Générale", "Capgemini", "Ecobank", "Bolloré Logistics", "CFAO", "Sopra Steria", "Canal+ Afrique", "BGFIBank", "Atos",
               "Ministère des Finances", "Mairie de Pointe-Noire", "Hôpital Général de Douala", "Lycée Savorgnan de Brazza", "ONG Médecins du Monde"],
        "en": ["Deloitte", "Unilever", "Standard Chartered", "IBM", "Accenture", "Barclays", "Shell", "Nestlé", "KPMG", "Microsoft", "Pfizer"]}
CITIES = {"fr": ["Brazzaville", "Pointe-Noire", "Kinshasa", "Douala", "Yaoundé", "Abidjan", "Dakar", "Libreville", "Paris", "Lyon",
                 "Marseille", "Lille", "Bordeaux", "Montréal", "Genève", "Lomé", "Cotonou", "Bamako", "Dolisie", "Lubumbashi"],
          "en": ["London", "Manchester", "New York", "Toronto", "Lagos", "Nairobi", "Accra", "Johannesburg", "Dublin"]}
TITLES = {"fr": ["Développeur Full-Stack", "Comptable", "Chargée de recrutement", "Chef de projet", "Data Analyst", "Technicien réseau",
                 "Assistante de direction", "Responsable logistique", "Ingénieur génie civil", "Infirmière", "Commercial terrain",
                 "Contrôleur de gestion", "Community manager", "Juriste d'affaires", "Caissière", "Enseignant de mathématiques", "Chauffeur-livreur"],
          "en": ["Software Engineer", "Accountant", "HR Officer", "Project Manager", "Data Analyst", "Sales Representative", "Nurse",
                 "Marketing Manager", "Financial Analyst", "Customer Service Agent"]}
HEAD = {"fr": ["EXPÉRIENCE PROFESSIONNELLE", "Expériences", "PARCOURS PROFESSIONNEL", "Expérience"], "en": ["WORK EXPERIENCE", "Experience", "EMPLOYMENT HISTORY"]}
CONNECT = {"fr": ["chez", "—", "|", "à", ",", "-"], "en": ["at", "—", "|", ",", "-"]}
FILL = {"fr": ["Gestion des dossiers clients et suivi des indicateurs.", "Encadrement d'une équipe de 5 personnes.", "Mise en place de procédures qualité."],
        "en": ["Managed client accounts and KPIs.", "Led a team of 5 people.", "Implemented quality procedures."]}

def synth_cv(lang=None):
    lang = lang or random.choice(["fr", "fr", "en"])
    f = fk[lang]
    parts, spans = [], []
    n = 0
    def add(text, label=None, upper=False):
        nonlocal n
        if upper: text = text.upper()
        toks = [t for t, _, _ in words_of(text)]
        if label and toks: spans.append([n, n + len(toks) - 1, label])
        parts.append(toks); n += len(toks)
    up = random.random() < 0.3
    add(f.name(), "person", upper=up)
    add(random.choice(TITLES[lang]), "job title", upper=random.random() < 0.2)
    if random.random() < 0.8:
        add(random.choice(["", "Adresse :", "Ville :", "📍"]) if lang == "fr" else ""); add(random.choice(CITIES[lang]), "city")
    add(random.choice(HEAD[lang]))
    for _ in range(random.randint(1, 3)):
        a, b = random.randint(2008, 2021), random.choice(["2023", "aujourd'hui" if lang == "fr" else "present", "2022"])
        order = random.random()
        if order < 0.5:
            add(random.choice(["•", "-", ""])); add(random.choice(TITLES[lang]), "job title"); add(random.choice(CONNECT[lang]))
            add(random.choice(ORGS[lang]), "organization"); add(f"({a} - {b})")
        else:
            add(f"{a} - {b} :"); add(random.choice(ORGS[lang]), "organization"); add(random.choice(CONNECT[lang]))
            add(random.choice(TITLES[lang]), "job title")
        if random.random() < 0.4: add(random.choice(CITIES[lang]), "city")
        add(random.choice(FILL[lang]))
    return {"tokenized_text": list(itertools.chain.from_iterable(parts)), "ner": spans}

synthetic = [synth_cv() for _ in range(3000)]
syn_test = [synth_cv() for _ in range(300)]
print(" ".join(synthetic[0]["tokenized_text"])[:300])

## 5. Score du modèle de base

In [ ]:
import torch
from gliner import GLiNER
device = "cuda" if torch.cuda.is_available() else "cpu"; print("device :", device)
base = GLiNER.from_pretrained(BASE_MODEL).to(device)
def f1(model, ds):
    _, f = model.evaluate(ds, flat_ner=True, entity_types=LABELS, threshold=0.4, batch_size=8)
    return float(f)
base_gold, base_syn = f1(base, gold), f1(base, syn_test)
print(f"Base : F1 CV réels={base_gold:.3f}  F1 synthétiques={base_syn:.3f}")

## 6. Vrais CV (Mehyaar, MIT) ré-annotés par le modèle de base — mémoire de ce qu'il sait déjà

In [ ]:
import unicodedata
z = zipfile.ZipFile(hf_hub_download("Mehyaar/Annotated_NER_PDF_Resumes", "ResumesJsonAnnotated.zip", repo_type="dataset", token=HF_TOKEN))
texts = [json.loads(z.read(n))["text"] for n in z.namelist() if n.endswith(".json")]
random.shuffle(texts)

def clean(t):
    # Textes extraits de PDF : puces privées (), largeur nulle, contrôles… font planter le tokenizer.
    t = "".join(c if c in "\n\t" or unicodedata.category(c)[0] not in "C" else " " for c in t)
    t = re.sub(r"[^\S\n]+", " ", t)
    return re.sub(r"\n\s*\n+", "\n", t).strip()

chunks = []
for text in texts[:1500]:
    text = clean(text)
    w = words_of(text)
    for s in range(0, min(len(w), 600), 250):
        c = w[s:s + 250]
        if len(c) >= 30:
            chunks.append(text[c[0][1]:c[-1][2]])
print(len(chunks), "extraits à annoter")

def to_spans(sub, ents):
    cw = words_of(sub)
    spans = []
    for e in ents:
        idx = [i for i, (_, a, b) in enumerate(cw) if a >= e["start"] and b <= e["end"]]
        if idx: spans.append([idx[0], idx[-1], e["label"]])
    return {"tokenized_text": [t for t, _, _ in cw], "ner": spans} if spans else None

pseudo, skipped = [], 0
from tqdm.auto import tqdm
for i in tqdm(range(0, len(chunks), 16)):
    batch = chunks[i:i + 16]
    try:   # par lots sur le GPU (rapide)
        preds = base.batch_predict_entities(batch, LABELS, threshold=0.75, flat_ner=True)
    except Exception:
        preds = []
        for sub in batch:   # repli extrait par extrait : un texte problématique est ignoré, pas bloquant
            try: preds.append(base.predict_entities(sub, LABELS, threshold=0.75, flat_ner=True))
            except Exception: preds.append([]); skipped += 1
    for sub, ents in zip(batch, preds):   # seulement les prédictions très sûres (≥ 0,75)
        ex = to_spans(sub, ents)
        if ex: pseudo.append(ex)
    if len(pseudo) >= 2500: break
print(len(pseudo), "extraits de vrais CV ré-annotés ;", skipped, "ignorés (texte illisible)")
del base; torch.cuda.empty_cache()

## 7. Fine-tuning (taux d'apprentissage bas : améliorer sans désapprendre)

In [ ]:
import gc
# Libère la mémoire GPU laissée par une tentative précédente ou par l'annotation (cellule 6).
for _v in ("trainer", "model", "base"):
    if _v in globals(): del globals()[_v]
gc.collect(); torch.cuda.empty_cache()

def windows(ex, size=120):
    """Découpe en fenêtres de ≤ 120 mots (≈ 250 sous-tokens) : tient dans la mémoire du T4."""
    toks, out = ex["tokenized_text"], []
    for s in range(0, len(toks), size):
        spans = [[a - s, b - s, l] for a, b, l in ex["ner"] if a >= s and b < s + size]
        if spans: out.append({"tokenized_text": toks[s:s + size], "ner": spans})
    return out

train = [w for ex in synthetic + pseudo for w in windows(ex)]
random.shuffle(train)
print(len(train), "exemples d'entraînement · longueur max :", max(len(x["tokenized_text"]) for x in train), "mots")

from gliner.training import Trainer, TrainingArguments
from gliner.data_processing.collator import DataCollator
model = GLiNER.from_pretrained(BASE_MODEL).to(device)
collator = DataCollator(model.config, data_processor=model.data_processor, prepare_labels=True)
args = TrainingArguments(
    output_dir="genesis-gliner", learning_rate=3e-6, weight_decay=0.01, others_lr=8e-6, others_weight_decay=0.01,
    lr_scheduler_type="linear", warmup_ratio=0.1,
    per_device_train_batch_size=4, gradient_accumulation_steps=2,   # lot effectif 8, moitié moins de mémoire
    per_device_eval_batch_size=4, max_grad_norm=1.0,
    focal_loss_alpha=0.75, focal_loss_gamma=2, num_train_epochs=2, eval_strategy="epoch", save_strategy="epoch",
    save_total_limit=2, dataloader_num_workers=0, fp16=(device == "cuda"), report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=train, eval_dataset=[w for ex in syn_test for w in windows(ex)],
                  tokenizer=model.data_processor.transformer_tokenizer, data_collator=collator)
print(f"Mémoire GPU libre : {torch.cuda.mem_get_info()[0] / 1e9:.1f} Go")
trainer.train()

## 8. Évaluation et publication (seulement si meilleur sur les CV réels)

In [ ]:
model.eval()
ft_gold, ft_syn = f1(model, gold), f1(model, syn_test)
print(f"Base       F1 CV réels={base_gold:.3f}  synthétiques={base_syn:.3f}")
print(f"Fine-tuné  F1 CV réels={ft_gold:.3f}  synthétiques={ft_syn:.3f}")
PUBLISHED = ft_gold >= base_gold + 0.01 and ft_syn >= base_syn - 0.01
if PUBLISHED:
    model.save_pretrained("genesis-gliner-cv")
    model.push_to_hub(TARGET_REPO, private=True, token=HF_TOKEN)
    print("✅ Publié (privé) :", TARGET_REPO)
else:
    print("❌ Pas d'amélioration sur les CV réels : le modèle de base reste en service (c'est un résultat valable).")

## 9. Export ONNX fp16 pour l'exécution locale dans Genesis

In [ ]:
# Export ONNX fp32 (la conversion fp16 d'onnxconverter-common casse un nœud Cast de DeBERTa ; fp32 = mêmes résultats, même vitesse sur CPU).
import os, shutil, glob, numpy as np, onnxruntime as ort
from huggingface_hub import HfApi
if not PUBLISHED:
    print("Rien à exporter.")
else:
    model.eval()
    m = model.cpu().float()
    text = "Aminata Diallo, Développeuse Full-Stack chez Orange à Brazzaville."
    inputs, _ = m.prepare_model_inputs([text], LABELS)
    names = ["input_ids", "attention_mask", "words_mask", "text_lengths", "span_idx", "span_mask"]

    class Wrapped(torch.nn.Module):
        """Entrées passées par NOM (l'ordre positionnel du modèle GLiNER diffère) ; sortie = logits seuls."""
        def __init__(self, core):
            super().__init__()
            self.core = core
        def forward(self, input_ids, attention_mask, words_mask, text_lengths, span_idx, span_mask):
            out = self.core(input_ids=input_ids, attention_mask=attention_mask, words_mask=words_mask,
                            text_lengths=text_lengths, span_idx=span_idx, span_mask=span_mask)
            return out.logits if hasattr(out, "logits") else out[0]

    wrapped = Wrapped(m.model).eval()
    shutil.rmtree("export", ignore_errors=True)
    os.makedirs("export/onnx", exist_ok=True)
    torch.onnx.export(
        wrapped, tuple(inputs[n] for n in names), "export/onnx/model.onnx",
        input_names=names, output_names=["logits"], opset_version=14, dynamo=False,
        dynamic_axes={"input_ids": {0: "batch_size", 1: "sequence_length"}, "attention_mask": {0: "batch_size", 1: "sequence_length"},
                      "words_mask": {0: "batch_size", 1: "sequence_length"}, "text_lengths": {0: "batch_size", 1: "value"},
                      "span_idx": {0: "batch_size", 1: "num_spans", 2: "idx"}, "span_mask": {0: "batch_size", 1: "num_spans"},
                      "logits": {0: "batch_size", 1: "sequence_length", 2: "num_spans", 3: "num_classes"}})
    print("Fichiers exportés :", [(f, round(os.path.getsize(f) / 1e6)) for f in glob.glob("export/onnx/*")])

    # Contrôle 1 : mêmes scores que PyTorch. Contrôle 2 : un texte plus long (dimensions dynamiques).
    sess = ort.InferenceSession("export/onnx/model.onnx", providers=["CPUExecutionProvider"])
    for t in [text, "Jean-Baptiste Kouka, Ingénieur travaux chez Colas Gabon à Libreville de 2017 à 2023, puis conducteur de travaux chez Sogea Satom."]:
        inp, _ = m.prepare_model_inputs([t], LABELS)
        with torch.no_grad():
            ref = torch.sigmoid(wrapped(*(inp[n] for n in names))).numpy()
        got = 1 / (1 + np.exp(-sess.run(["logits"], {n: inp[n].numpy() for n in names})[0]))
        diff = float(np.abs(ref - got).max())
        print(f"Écart max PyTorch / ONNX : {diff:.5f}")
        assert diff < 0.01, "Export ONNX incohérent : non publié"

    shutil.copy(hf_hub_download("onnx-community/gliner_multi-v2.1", "tokenizer.json"), "export/tokenizer.json")
    shutil.copy("genesis-gliner-cv/gliner_config.json", "export/gliner_config.json")
    api = HfApi(token=HF_TOKEN)
    api.create_repo(TARGET_REPO + "-onnx", private=True, exist_ok=True)
    api.upload_folder(repo_id=TARGET_REPO + "-onnx", folder_path="export")
    print("✅ ONNX publié (privé) :", TARGET_REPO + "-onnx")